In [1]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so saconv_backbone.py can be imported

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 4.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.137 | torch 2.11.0+cu128 | torchvision 0.26.0+cu128
GPU: Tesla T4


In [2]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to digitalaccess-2(copy)-4 in yolo26:: 100%|██████████| 17613/17613 [00:01<00:00, 10504.15it/s]


train : 7704 images
valid : 550 images
test  : 550 images
nc = 27
✅ Dataset prepared!


In [3]:
%%writefile saconv_backbone.py
"""Paper 3 backbone for YOLO26n: stride-2 Conv at layers 3, 5, 7 -> SAConv (Switchable Atrous Convolution)."""
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

SACONV_LAYERS = (3, 5, 7)


class SAConv(nn.Module):
    """Switchable Atrous Convolution: shared 3x3 weight at dilation 1 and 3, mixed by a learned switch, with pre/post global context."""

    def __init__(self, c1, c2, k=3, s=2, rates=(1, 3)):
        super().__init__()
        self.c1, self.c2, self.k, self.s, self.rates = c1, c2, k, s, rates
        self.pre_context = nn.Conv2d(c1, c1, 1, bias=True)
        self.post_context = nn.Conv2d(c2, c2, 1, bias=True)
        for m in (self.pre_context, self.post_context):  # start as identity
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)
        self.switch = nn.Conv2d(c1, 1, 1, stride=s, bias=True)
        nn.init.zeros_(self.switch.weight)
        nn.init.zeros_(self.switch.bias)  # sigmoid(0) = 0.5: both branches equal at the start
        self.weight = nn.Parameter(torch.empty(c2, c1, k, k))
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)
        self.weight_diff = nn.Parameter(torch.zeros(c2, c1, k, k))  # extra weight of the large-rate branch
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()
        self.static = False

    def to_static(self):
        """Replace the shared-weight branches by two plain Conv2d layers (same output), for NCNN export."""
        if self.static:
            return
        r1, r2 = self.rates
        p = self.k // 2
        self.conv_small = nn.Conv2d(self.c1, self.c2, self.k, self.s, p * r1, dilation=r1, bias=False)
        self.conv_large = nn.Conv2d(self.c1, self.c2, self.k, self.s, p * r2, dilation=r2, bias=False)
        with torch.no_grad():
            self.conv_small.weight.copy_(self.weight)
            self.conv_large.weight.copy_(self.weight + self.weight_diff)
        self.conv_small.to(self.weight.device, self.weight.dtype)
        self.conv_large.to(self.weight.device, self.weight.dtype)
        del self.weight, self.weight_diff
        self.static = True

    def forward(self, x):
        x = x + self.pre_context(F.adaptive_avg_pool2d(x, 1))
        s = torch.sigmoid(self.switch(F.avg_pool2d(x, 5, 1, 2)))
        if self.static:
            small, large = self.conv_small(x), self.conv_large(x)
        else:
            r1, r2 = self.rates
            p = self.k // 2
            small = F.conv2d(x, self.weight, None, self.s, p * r1, r1)
            large = F.conv2d(x, self.weight + self.weight_diff, None, self.s, p * r2, r2)
        y = s * small + (1 - s) * large
        y = y + self.post_context(F.adaptive_avg_pool2d(y, 1))
        return self.act(self.bn(y))


def add_saconv(model, layers=SACONV_LAYERS):
    """Replace the stride-2 Conv at the given layers with SAConv, starting from that Conv's (pretrained) weight and BN."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, Conv) and old.conv.kernel_size == (3, 3) and old.conv.stride == (2, 2), f"layer {i} is not a 3x3 s2 Conv"
        new = SAConv(old.conv.in_channels, old.conv.out_channels, 3, 2)
        with torch.no_grad():
            new.weight.copy_(old.conv.weight)
        new.bn.load_state_dict(old.bn.state_dict())
        new.i, new.f, new.type = old.i, old.f, "SAConv"  # keep the routing attributes used by the model's forward
        new.np = sum(p.numel() for p in new.parameters())
        model.model[i] = new
    return model


def to_static(model):
    """Convert every SAConv in the model to its deploy form. Returns the number of converted layers."""
    n = 0
    for m in model.modules():
        if isinstance(m, SAConv) and not m.static:
            m.to_static()
            n += 1
    return n


class SAConvTrainer(DetectionTrainer):
    """Load the pretrained weights into the base model first, then insert SAConv initialised from those weights."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_saconv(model)
        if RANK in {-1, 0}:
            print(f"Paper 3 backbone: SAConv at layers {SACONV_LAYERS}")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model

Writing saconv_backbone.py


In [4]:
import importlib, saconv_backbone
importlib.reload(saconv_backbone)
from saconv_backbone import add_saconv, to_static, SAConvTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    """Params and GFLOPs of the deploy model (SAConv in static form, Conv+BN fused)."""
    m = deepcopy(model).float().cpu().eval()
    to_static(m)
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs Paper 3 backbone
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False)
sac = add_saconv(DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False))
sac.eval()
x = torch.rand(1, 3, 640, 640)
with torch.no_grad():
    y_train_form = sac(x)[0]
    stat = deepcopy(sac); to_static(stat)
    y_static = stat(x)[0]
print("layers 3, 5, 7:", [type(sac.model[i]).__name__ for i in (3, 5, 7)])
print("static form gives the same output:", torch.allclose(y_train_form, y_static, atol=1e-4))
for tag, m in (("base", base), ("Paper 3 backbone", sac)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:17s} Params {pm:.3f} M | GFLOPs {gf:.2f}")

Overriding model.yaml nc=80 with nc=27
Overriding model.yaml nc=80 with nc=27
layers 3, 5, 7: ['SAConv', 'SAConv', 'SAConv']
static form gives the same output: True
base              Params 2.380 M | GFLOPs 5.34
Paper 3 backbone  Params 2.984 M | GFLOPs 6.52


In [5]:
from ultralytics import YOLO

EPOCHS  = 100      # Step A = 1 (export + Pi speed check) | Step B = 100 (real training)
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("SAConv Backbone (Paper 3)", "bb_saconv"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

print(f"Step {'A (1 epoch: export + speed check)' if EPOCHS == 1 else 'B (full training)'}")
for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=SAConvTrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

Step B (full training)

========== SAConv Backbone (Paper 3) (bb_saconv) ==========
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/digitalaccess/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, 

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      3.34G      1.343      2.184    0.01811         14        640: 100% ━━━━━━━━━━━━ 482/482 4.2it/s 1:55
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.3it/s 2.5s
                   all        550        557      0.723       0.81      0.874      0.501

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      3.35G      1.404      1.454    0.01904         12        640: 100% ━━━━━━━━━━━━ 482/482 4.3it/s 1:52
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.0it/s 2.6s
                   all        550        557      0.823       0.86      0.918      0.544

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100      3.35G      1.441       1.16    0.01949          7        640: 100% ━━━━━━━━━━━━ 482/482 4.3it/s 1:51
                 Class     Images  Instances      Box(P 

In [6]:
import saconv_backbone  # must be imported before loading an SAConv checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
print(f"(trained for {EPOCHS} epoch{'s' if EPOCHS > 1 else ''})")
METRICS.drop(columns="run")

Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 134 layers, 2,983,752 parameters, 0 gradients, 4.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 977.6±281.7 MB/s, size: 37.7 KB)
val: Scanning /kaggle/working/digitalaccess/test/labels... 550 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 1.3Kit/s 0.4s
val: New cache created: /kaggle/working/digitalaccess/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 12.9it/s 2.7s
                   all        550        552      0.964      0.941      0.961      0.581
Speed: 0.5ms preprocess, 2.9ms inference, 0.0ms loss, 0.2ms postprocess per image
(trained for 100 epochs)


,Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50
0,SAConv Backbone (Paper 3),2.98,6.52,0.964,0.941,0.952,0.961


In [7]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper3_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("saconv_backbone.py", EDGE / "saconv_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

from saconv_backbone import to_static

NCNN_OK = False
for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")  # Jetson: the trained model as is
    print(f"{run:12s} -> {run}.pt")
    try:
        yolo = YOLO(str(w))
        n = to_static(yolo.model)  # shared-weight branches -> two plain convs, so NCNN sees standard layers
        p = yolo.export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
        shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
        NCNN_OK = True
        print(f"\n✅ NCNN export OK ({n} SAConv layers converted) -> {run}_ncnn_model/  -> run pi5_bench.py on the Pi")
    except Exception as e:
        NCNN_OK = False
        print(f"\n❌ NCNN export FAILED: {type(e).__name__}: {e}")
        print("   SAConv cannot run on the Raspberry Pi 5 with NCNN -> drop it (Jetson package is still created).")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import saconv_backbone  # needed to load the SAConv checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper3_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper3_backbone_edge_bench.zip from the Output tab.")

bench_images: 50 real test images
bb_saconv    -> bb_saconv.pt
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This export format does not support end2end models, disabling the end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 164 layers, 3,108,485 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from '/kaggle/working/runs/bb_saconv_s0/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 31, 8400) (6.3 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 1 package in 196ms
 Downloaded ncnn
Prepared 1 package in 124ms
Installed 1 package in 3ms
 + ncnn==1.0.20261020

requirements: AutoUpdate success ✅ 0

pnnxparam = /kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = saconv_backbone.SAConv
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bottleneck
i

NCNN: export success ✅ 7.2s, saved as '/kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model' (11.6 MB)

Export complete (7.8s)
Results saved to /kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/bb_saconv_s0/weights/best_ncnn_model imgsz=640 data=/kaggle/working/digitalaccess/data.yaml  
Visualize:       https://netron.app

✅ NCNN export OK (3 SAConv layers converted) -> bb_saconv_ncnn_model/  -> run pi5_bench.py on the Pi

Download paper3_backbone_edge_bench.zip from the Output tab.


In [8]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_saconv": None}
FPS_JETSON = {"bb_saconv": None}

if EPOCHS == 1:
    print("⚠️ This is the 1-epoch step A model: accuracy values are not final.")
table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50,FPS (Rasp-5),FPS (Jetson)
Standard YOLO26n Backbone,2.380000,5.340000,0.971000,0.937000,0.953000,0.958000,13.380000,--
SAConv Backbone (Paper 3),2.980000,6.520000,0.964000,0.941000,0.952000,0.961000,--,--


In [9]:
import zipfile

ALL_ZIP = WORK / "paper3_backbone_all_results.zip"
folders = [WORK / "paper3_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")

paper3_backbone_edge_bench/    59 files
runs/                          29 files
weights/bb_saconv_best.pt

Download paper3_backbone_all_results.zip (52.2 MB) from the Output tab.
